# Imports and Setup

In [1]:
# Imports
import os
import dotenv
import re
import pandas as pd
from googleapiclient.discovery import build
from youtube_transcript_api import YouTubeTranscriptApi
from datetime import datetime, timezone

In [6]:
# Load environment variables
dotenv.load_dotenv()

True

In [7]:
# Get the API key 
API_KEY = os.getenv("YOUTUBE_API_KEY")

# Set up the YouTube Data API client 
youtube = build("youtube", "v3", developerKey=API_KEY)

In [8]:
# Define the channel ID
SKZ_CHANNEL_ID = "UC9rMiEjNaCSsebs31MRDCRA"

# Map the base Channel ID to the specific hidden playlists
base_id = SKZ_CHANNEL_ID.replace("UC", "")
playlists_to_process = {
    "Long-form": f"UULF{base_id}",
    "Short": f"UUSH{base_id}",
    "Live/VOD": f"UULV{base_id}"
}

In [9]:
playlists_to_process.get("Long-form")

'UULF9rMiEjNaCSsebs31MRDCRA'

# Fetching Video IDs

In [ ]:
# Fetch Longform Video IDs via Pagination
# For testing, limit this to just 2 pages (2 x 50 videos)
longform_playlist_id = playlists_to_process.get("Long-form")

video_ids = []
next_page_token = None

response_1 = youtube.playlistItems().list(
    part="contentDetails",
    playlistId=longform_playlist_id,
    maxResults=50,
    pageToken=next_page_token
).execute()
display(response_1)

{'kind': 'youtube#playlistItemListResponse',
 'etag': 'ejzbVvl0_KLUTybNZ5Qf6qAj7RY',
 'nextPageToken': 'EAAaHlBUOkNESWlFRFpCUVRFMU9UTTJNalpDTkRaRU5EVQ',
 'items': [{'kind': 'youtube#playlistItem',
   'etag': 'Pgpq1cPtVhIWHDtLy4ciXHRghyE',
   'id': 'VVVMRjlyTWlFak5hQ1NzZWJzMzFNUkRDUkEubDhzU1FBYTFrTk0',
   'contentDetails': {'videoId': 'l8sSQAa1kNM',
    'videoPublishedAt': '2026-03-19T15:00:08Z'}},
  {'kind': 'youtube#playlistItem',
   'etag': 'oFU16bh108UH8TWcJCCC9sveSiY',
   'id': 'VVVMRjlyTWlFak5hQ1NzZWJzMzFNUkRDUkEuOURTT0Z5TFN2NlE',
   'contentDetails': {'videoId': '9DSOFyLSv6Q',
    'videoPublishedAt': '2026-03-19T10:59:58Z'}},
  {'kind': 'youtube#playlistItem',
   'etag': '10XJoKMDlGcDgkkTKbhB6EmkAUI',
   'id': 'VVVMRjlyTWlFak5hQ1NzZWJzMzFNUkRDUkEuSmw1bk5xYWpyVjA',
   'contentDetails': {'videoId': 'Jl5nNqajrV0',
    'videoPublishedAt': '2026-03-18T10:59:56Z'}},
  {'kind': 'youtube#playlistItem',
   'etag': 'E--JGQz9mGk_qs9A6DTov7_GU5k',
   'id': 'VVVMRjlyTWlFak5hQ1NzZWJzMzFNUkRDUk

In [7]:
# Get the list of playllist items
print(len(response_1["items"]), " values")
display(response_1["items"])

50  values


[{'kind': 'youtube#playlistItem',
  'etag': 'Pgpq1cPtVhIWHDtLy4ciXHRghyE',
  'id': 'VVVMRjlyTWlFak5hQ1NzZWJzMzFNUkRDUkEubDhzU1FBYTFrTk0',
  'contentDetails': {'videoId': 'l8sSQAa1kNM',
   'videoPublishedAt': '2026-03-19T15:00:08Z'}},
 {'kind': 'youtube#playlistItem',
  'etag': 'oFU16bh108UH8TWcJCCC9sveSiY',
  'id': 'VVVMRjlyTWlFak5hQ1NzZWJzMzFNUkRDUkEuOURTT0Z5TFN2NlE',
  'contentDetails': {'videoId': '9DSOFyLSv6Q',
   'videoPublishedAt': '2026-03-19T10:59:58Z'}},
 {'kind': 'youtube#playlistItem',
  'etag': '10XJoKMDlGcDgkkTKbhB6EmkAUI',
  'id': 'VVVMRjlyTWlFak5hQ1NzZWJzMzFNUkRDUkEuSmw1bk5xYWpyVjA',
  'contentDetails': {'videoId': 'Jl5nNqajrV0',
   'videoPublishedAt': '2026-03-18T10:59:56Z'}},
 {'kind': 'youtube#playlistItem',
  'etag': 'E--JGQz9mGk_qs9A6DTov7_GU5k',
  'id': 'VVVMRjlyTWlFak5hQ1NzZWJzMzFNUkRDUkEuMXRpcGsyWDFhVW8',
  'contentDetails': {'videoId': '1tipk2X1aUo',
   'videoPublishedAt': '2026-03-17T10:59:56Z'}},
 {'kind': 'youtube#playlistItem',
  'etag': 'nM2yJxVNY4muDJyiuE0

In [8]:
response_1["items"][0]["contentDetails"]["videoId"]

'l8sSQAa1kNM'

In [9]:
[item["contentDetails"] for item in response_1["items"]]

[{'videoId': 'l8sSQAa1kNM', 'videoPublishedAt': '2026-03-19T15:00:08Z'},
 {'videoId': '9DSOFyLSv6Q', 'videoPublishedAt': '2026-03-19T10:59:58Z'},
 {'videoId': 'Jl5nNqajrV0', 'videoPublishedAt': '2026-03-18T10:59:56Z'},
 {'videoId': '1tipk2X1aUo', 'videoPublishedAt': '2026-03-17T10:59:56Z'},
 {'videoId': 'bDbzZ_Dx4Ks', 'videoPublishedAt': '2026-03-15T11:00:18Z'},
 {'videoId': 'uB-XPe1K7SU', 'videoPublishedAt': '2026-03-12T10:59:56Z'},
 {'videoId': 'X7jAms_f4Rc', 'videoPublishedAt': '2026-03-10T11:00:03Z'},
 {'videoId': 'bSZL3ZZC8EE', 'videoPublishedAt': '2026-03-07T15:00:00Z'},
 {'videoId': 'o66KJX1YJ3s', 'videoPublishedAt': '2026-03-05T10:59:57Z'},
 {'videoId': 'ZV5IjNgpBFQ', 'videoPublishedAt': '2026-03-04T10:59:58Z'},
 {'videoId': 'sKi1gEtnSAQ', 'videoPublishedAt': '2026-03-04T06:25:03Z'},
 {'videoId': 'x-kl7fzEmaY', 'videoPublishedAt': '2026-02-28T10:59:59Z'},
 {'videoId': 'h1li3NlTB0I', 'videoPublishedAt': '2026-02-26T10:59:58Z'},
 {'videoId': 'ODoEa_1gPSc', 'videoPublishedAt': '20

In [10]:
# Append the first 50 video IDs to video_ids
vid_ids_1 = [item["contentDetails"]["videoId"] for item in response_1["items"]]
video_ids.extend(vid_ids_1)
display(video_ids)

['l8sSQAa1kNM',
 '9DSOFyLSv6Q',
 'Jl5nNqajrV0',
 '1tipk2X1aUo',
 'bDbzZ_Dx4Ks',
 'uB-XPe1K7SU',
 'X7jAms_f4Rc',
 'bSZL3ZZC8EE',
 'o66KJX1YJ3s',
 'ZV5IjNgpBFQ',
 'sKi1gEtnSAQ',
 'x-kl7fzEmaY',
 'h1li3NlTB0I',
 'ODoEa_1gPSc',
 'HFbirADU3_Q',
 'FEDQ14t58CA',
 'mUFKUJTMjWg',
 'PUUxE7EGP4Q',
 'OSUHqAvuSRI',
 'F6lf26nWk10',
 'Tsr3lBEn9ww',
 'IiqoozFOs1s',
 'v1xm6Pck-mo',
 'XOcH3lgw0KE',
 'bpnab0Aa5mE',
 'fETjgY83o7Q',
 'urNLPgalt6o',
 '9bHJc5Hv9Vw',
 '5Oio145WBB8',
 'nv6WtNmPXZU',
 'YsvbN665LZE',
 'ZeoHs2zfXPE',
 'wgfPkWSDUHU',
 'MI6_aH1JT7s',
 'HdgpH3_mZEs',
 'dxi4pB9b7eg',
 'uRJUVUlTevM',
 'PET396MCIV8',
 'TA7C2NBYw88',
 '3z27jOgXx-I',
 '5bzHeSphgUU',
 'ZqnBCh51qUw',
 'QNKTZSZ-X3s',
 'FR99OlxOVus',
 'xqu9eeGQxXc',
 '7oxJSrN2CNQ',
 'vPPEfLv1EpE',
 'k4IsJDwRKz4',
 'udFiiR4V6mg',
 'wIJ2TcnDKPs']

In [11]:
# Fetch the last 50 longform video IDs
next_page_token = response_1.get("nextPageToken")

response_2 = youtube.playlistItems().list(
    part="contentDetails",
    playlistId=longform_playlist_id,
    maxResults=50,
    pageToken=next_page_token
).execute()

vid_ids_2 = [item["contentDetails"]["videoId"] for item in response_2["items"]]
video_ids.extend(vid_ids_2)

In [12]:
# Total of the video IDs collected so far
len(video_ids)

100

In [15]:
video_ids[0:2]

['l8sSQAa1kNM', '9DSOFyLSv6Q']

In [11]:
all_current_videos = []
video_format = "Long-form"

for vid in video_ids:
    all_current_videos.append({"id": vid, "format": video_format})

In [12]:
all_current_videos

[{'id': 'l8sSQAa1kNM', 'format': 'Long-form'},
 {'id': '9DSOFyLSv6Q', 'format': 'Long-form'},
 {'id': 'Jl5nNqajrV0', 'format': 'Long-form'},
 {'id': '1tipk2X1aUo', 'format': 'Long-form'},
 {'id': 'bDbzZ_Dx4Ks', 'format': 'Long-form'},
 {'id': 'uB-XPe1K7SU', 'format': 'Long-form'},
 {'id': 'X7jAms_f4Rc', 'format': 'Long-form'},
 {'id': 'bSZL3ZZC8EE', 'format': 'Long-form'},
 {'id': 'o66KJX1YJ3s', 'format': 'Long-form'},
 {'id': 'ZV5IjNgpBFQ', 'format': 'Long-form'},
 {'id': 'sKi1gEtnSAQ', 'format': 'Long-form'},
 {'id': 'x-kl7fzEmaY', 'format': 'Long-form'},
 {'id': 'h1li3NlTB0I', 'format': 'Long-form'},
 {'id': 'ODoEa_1gPSc', 'format': 'Long-form'},
 {'id': 'HFbirADU3_Q', 'format': 'Long-form'},
 {'id': 'FEDQ14t58CA', 'format': 'Long-form'},
 {'id': 'mUFKUJTMjWg', 'format': 'Long-form'},
 {'id': 'PUUxE7EGP4Q', 'format': 'Long-form'},
 {'id': 'OSUHqAvuSRI', 'format': 'Long-form'},
 {'id': 'F6lf26nWk10', 'format': 'Long-form'},
 {'id': 'Tsr3lBEn9ww', 'format': 'Long-form'},
 {'id': 'Iiqo

# Fetching Video Metadata & Stats

In [20]:
# Batch Process Videos for Stats & Metadata (2 at a time)
chunk_size = 2

batch = all_current_videos[0: 0 + chunk_size]
display(batch)
batch_ids = [v["id"] for v in batch]
display(batch_ids)
batch_formats = {v["id"]: v["format"] for v in batch}
display(batch_formats)

[{'id': 'l8sSQAa1kNM', 'format': 'Long-form'},
 {'id': '9DSOFyLSv6Q', 'format': 'Long-form'}]

['l8sSQAa1kNM', '9DSOFyLSv6Q']

{'l8sSQAa1kNM': 'Long-form', '9DSOFyLSv6Q': 'Long-form'}

In [24]:
",".join(batch_ids)

'l8sSQAa1kNM,9DSOFyLSv6Q'

In [ ]:
# Extract Stats for Two Videos
vid_response = youtube.videos().list(
    part="snippet,statistics",
    id=batch_ids
).execute()
display(vid_response)

{'kind': 'youtube#videoListResponse',
 'etag': 'qFaiM4CH5faIvtUtVOYQ0G20VWw',
 'items': [{'kind': 'youtube#video',
   'etag': 'kYeYLcSnn9SLwyAGyLX3wCEiGmk',
   'id': 'l8sSQAa1kNM',
   'snippet': {'publishedAt': '2026-03-19T15:00:08Z',
    'channelId': 'UC9rMiEjNaCSsebs31MRDCRA',
    'title': 'Hyunjin "LOVER" Video',
    'description': '현진(Hyunjin) "LOVER" Video\n\n[Credits]\nLyrics by : 현진\nComposed by : 현진, joha\nArranged by : joha\nOriginal Publisher : JYP Entertainment, Copyright Control\n\n[Hyunjin "LOVER" Lyrics]\nYeah yeah yeah yeah\nI’m running in the rain yeah\n망가진 내 맘 씻겨 내리길\n한 때 사랑하긴 했던 거지? Right, right?\n\nLover lover lover is hurt ’cause she went away \nLoved her loved her loved her so much until that rainy day\nLover lover lover dies ’cause there’s no way\nLover lover love is gone no matter what I say\n \n오 맨정신은 벅차\nI don’t want to drink no more no more no more\nWhy’re you acting like it’s me \nIt’s all my fault, you mean?\n하긴 원래 넌 그런 애였을 테니\n \nWhy why is it not me\nIn yo

In [17]:
type(vid_response['items'])

list

In [22]:
vid_response['items'][0]

{'kind': 'youtube#video',
 'etag': 'kYeYLcSnn9SLwyAGyLX3wCEiGmk',
 'id': 'l8sSQAa1kNM',
 'snippet': {'publishedAt': '2026-03-19T15:00:08Z',
  'channelId': 'UC9rMiEjNaCSsebs31MRDCRA',
  'title': 'Hyunjin "LOVER" Video',
  'description': '현진(Hyunjin) "LOVER" Video\n\n[Credits]\nLyrics by : 현진\nComposed by : 현진, joha\nArranged by : joha\nOriginal Publisher : JYP Entertainment, Copyright Control\n\n[Hyunjin "LOVER" Lyrics]\nYeah yeah yeah yeah\nI’m running in the rain yeah\n망가진 내 맘 씻겨 내리길\n한 때 사랑하긴 했던 거지? Right, right?\n\nLover lover lover is hurt ’cause she went away \nLoved her loved her loved her so much until that rainy day\nLover lover lover dies ’cause there’s no way\nLover lover love is gone no matter what I say\n \n오 맨정신은 벅차\nI don’t want to drink no more no more no more\nWhy’re you acting like it’s me \nIt’s all my fault, you mean?\n하긴 원래 넌 그런 애였을 테니\n \nWhy why is it not me\nIn your eyes 이유를 찾지 ooh ooh ooh ooh\n\nYeah yeah yeah yeah\nI’m running in the rain yeah\n망가진 내 맘 씻겨 내리길\n

In [21]:
vid_response['items'][1]

{'kind': 'youtube#video',
 'etag': 'hZnX-8X9m4IilVs0Ay9Oqh6wnlw',
 'id': '9DSOFyLSv6Q',
 'snippet': {'publishedAt': '2026-03-19T10:59:58Z',
  'channelId': 'UC9rMiEjNaCSsebs31MRDCRA',
  'title': '스키즈 문화센터 (SKZ Community Center) #2｜[SKZ CODE] Ep.94',
  'description': '스키즈 문화센터 (SKZ Community Center) #2｜[SKZ CODE(스키즈 코드)] Ep.94\n\n✨ English, Japanese, Chinese, Spanish, Indonesian & Thai subtitles are available!\n\nListen to "DO IT" now☁️\nhttps://Stray-Kids.lnk.to/DOIT\n\nListen to "Do It (Remixes)" now🎊\nhttps://Stray-Kids.lnk.to/DoItRemixes\n\nStray Kids "DO IT"\niTunes & Apple Music: https://Stray-Kids.lnk.to/DOIT/AppleMusic\nSpotify: https://Stray-Kids.lnk.to/DOIT/Spotify\n\nStray Kids "Do It (Remixes)"\niTunes & Apple Music: https://Stray-Kids.lnk.to/DoItRemixes/AppleMusic\nSpotify: https://Stray-Kids.lnk.to/DoItRemixes/Spotify\n\nStray Kids Official YouTube: https://www.youtube.com/c/StrayKids\nStray Kids Official Twitter: https://twitter.com/Stray_Kids\nStray Kids Official Instagra

In [25]:
# Get the Video Stats
vid_info = vid_response['items'][0]
title = vid_info['snippet']['title']
stats = vid_info['statistics']

print(f"Title: {title}")
print(f"Views: {stats.get('viewCount')}")

Title: Hyunjin "LOVER" Video
Views: 5628876


In [27]:
vid_info.get("statistics", {})

{'viewCount': '5628876',
 'likeCount': '705144',
 'favoriteCount': '0',
 'commentCount': '53486'}

# Fetching Top Video Comments 

In [18]:
# Extract Top Comments for One Video
comment_response = youtube.commentThreads().list(
        part="snippet",
        videoId=video_ids[0],
        maxResults=5, # Just 5 for testing
        order="relevance", 
        textFormat="plainText"
    ).execute()
display(comment_response)

{'kind': 'youtube#commentThreadListResponse',
 'etag': 'mrmH9XCu-1fkPZouSe4elV12-WI',
 'nextPageToken': 'Z2V0X3JhbmtlZF9zdHJlYW1zLS1Da1VJZ0FRVkY3ZlJPQm83Q2pjSTJGOFFnQVFZQnlJdEYxX3dwS1RRQTZPMk40WE41bzlyQWxSZ29TbEI2RmtST2t3ZVBycmlYY0d5OHpUbjczM3RvZGNPM0lNQkVBVVNCUWlvSUJnQUVnVUloeUFZQUJJRkNJa2dHQUFTQlFpSUlCZ0FFZ2NJaENBUUJSZ0JFZ1VJdXlBWUFCSUhDSVVnRUFFWUFR',
 'pageInfo': {'totalResults': 5, 'resultsPerPage': 5},
 'items': [{'kind': 'youtube#commentThread',
   'etag': '6mCl7JPmlraWmSn66bQKe3eMtX0',
   'id': 'UgxzqAIYGVrwVt2FLJ94AaABAg',
   'snippet': {'channelId': 'UC9rMiEjNaCSsebs31MRDCRA',
    'videoId': 'l8sSQAa1kNM',
    'topLevelComment': {'kind': 'youtube#comment',
     'etag': 'tO8ScPftDqfY9dZazHClTjdzOFQ',
     'id': 'UgxzqAIYGVrwVt2FLJ94AaABAg',
     'snippet': {'channelId': 'UC9rMiEjNaCSsebs31MRDCRA',
      'videoId': 'l8sSQAa1kNM',
      'textDisplay': 'Haters: Hyunjin can’t sing\nHyunjin: Hold my paintbrush-',
      'textOriginal': 'Haters: Hyunjin can’t sing\nHyunjin: Hold my pa

In [19]:
# Print The Top Comments
for item in comment_response.get('items', []):
    comment_text = item['snippet']['topLevelComment']['snippet']['textDisplay']
    # Cutting off at 100 characters for clean printing
    print(f"- {comment_text[:100]}...")

- Haters: Hyunjin can’t sing
Hyunjin: Hold my paintbrush-...
- WE NEED THIS ON SPOTIFY...
- Hyunjin didn’t just release a video; he released a feeling.
This is art....
- #15 on youtube music chart. 
4 million in 4 days. 
Zero promotion. 
Damn Hyunjin you killed it 🫶...
- his voice is so different here but it's insane!!...


In [ ]:
# Put it into a DataFrame
# If we had run a loop over all videos, we'd have a list of dictionaries like this:
mock_video_data = [{
    "video_id": video_ids[0],
    "video_format": "Long-form",
    "title": title,
    "view_count": int(stats.get("viewCount", 0))
}]

df_videos = pd.DataFrame(mock_video_data)
display(df_videos)

,video_id,video_format,title,content_pillar,view_count
0,l8sSQAa1kNM,Long-form,"Hyunjin ""LOVER"" Video",Other/Misc,4422170


In [ ]:
# Fetch Stats for Only 100 Longform Videos to Analyze the Titles
sample_video_data = []

for video_id in video_ids:
    response = youtube.videos().list(
        part="snippet,statistics",
        id=video_id
    ).execute()
    
    snippet = response['items'][0]['snippet']
    stats = response['items'][0].get('statistics', {})
    
    sample_video_data.append({
        'video_id': video_id,
        'published_date': snippet['publishedAt'],
        'title': snippet['title'],
        'view_count': int(stats.get('viewCount', 0)),
        'like_count': int(stats.get('likeCount', 0)),
        'comment_count': int(stats.get('commentCount', 0)),
        'description': snippet['description'],
        'category_id': snippet['categoryId'],
        'tags': ",".join(snippet.get('tags', []))
    })

In [36]:
df_videos = pd.DataFrame(sample_video_data)

In [29]:
df_videos.loc[1, 'title']

'스키즈 문화센터 (SKZ Community Center) #2｜[SKZ CODE] Ep.94'

In [37]:
df_videos.head(10)

,video_id,published_date,title,content_pillar,view_count,like_count,comment_count,description,category_id,tags
0,l8sSQAa1kNM,2026-03-19T15:00:08Z,"Hyunjin ""LOVER"" Video",Other/Misc,4458919,672927,51762,"현진(Hyunjin) ""LOVER"" Video\n\n[Credits]\nLyrics...",10,"JYP Entertainment,JYP,Stray Kids,스트레이 키즈,SKZ,스..."
1,9DSOFyLSv6Q,2026-03-19T10:59:58Z,스키즈 문화센터 (SKZ Community Center) #2｜[SKZ CODE] ...,SKZ CODE,1752250,176497,6841,스키즈 문화센터 (SKZ Community Center) #2｜[SKZ CODE(스...,10,"Stray Kids,스트레이 키즈,SKZ,스키즈,방찬,BANG CHAN,리노,이민호..."
2,Jl5nNqajrV0,2026-03-18T10:59:56Z,[Stray Kids : SKZ-TALKER] Ep.81,SKZ-TALKER,563802,86181,3660,[Stray Kids(스트레이 키즈) : SKZ-TALKER(슼즈토커)] Ep.81...,10,"JYP Entertainment,JYP,Stray Kids,스트레이 키즈,SKZ,스..."
3,1tipk2X1aUo,2026-03-17T10:59:56Z,[RACHA LOG] Ep.17 청담꿀즈 : Bang Chan X Changbin ...,Other/Misc,711271,104361,3581,[RACHA LOG(라차로그)] Ep.17 청담꿀즈 : 방찬 X 창빈 X 승민 (B...,10,"Stray Kids,스트레이 키즈,SKZ,방찬,BANG CHAN,리노,이민호,LEE..."
4,bDbzZ_Dx4Ks,2026-03-15T11:00:18Z,Seungmin - 달리기 (윤상 Cover) | [SONG by 2] S#1,Other/Misc,741007,151678,10295,승민(Seungmin) - 달리기 (윤상 Cover) | [SONG by 2(송 바...,10,"Stray Kids,스트레이 키즈,SKZ,스키즈,방찬,BANG CHAN,리노,이민호..."
5,uB-XPe1K7SU,2026-03-12T10:59:56Z,스키즈 문화센터 (SKZ Community Center) #1｜[SKZ CODE] ...,SKZ CODE,2796113,264264,10108,스키즈 문화센터 (SKZ Community Center) #1｜[SKZ CODE(스...,10,"Stray Kids,스트레이 키즈,SKZ,스키즈,방찬,BANG CHAN,리노,이민호..."
6,X7jAms_f4Rc,2026-03-10T11:00:03Z,[RACHA LOG] Ep.16 쓰리라차 : Bang Chan X Changbin ...,Other/Misc,828258,117942,3586,[RACHA LOG(라차로그)] Ep.16 쓰리라차 : 방찬 X 창빈 X 한 (Ba...,10,"Stray Kids,스트레이 키즈,SKZ,방찬,BANG CHAN,리노,이민호,LEE..."
7,bSZL3ZZC8EE,2026-03-07T15:00:00Z,"I.N ""행복 뭐 별거 없네요(The Little Things)"" | [Stray ...",SKZ-PLAYER/RECORD,1081033,195744,12025,"아이엔(I.N) ""행복 뭐 별거 없네요(The Little Things)"" | [S...",10,"Stray Kids,스트레이 키즈,SKZ,방찬,BANG CHAN,리노,이민호,LEE..."
8,o66KJX1YJ3s,2026-03-05T10:59:57Z,"Stray Kids ""DO IT"" Recording Scene | Do It, 신선...",Other/Misc,1503301,180796,5425,"Stray Kids(스트레이 키즈) ""DO IT"" Recording Scene | ...",10,"Stray Kids,스트레이 키즈,SKZ,방찬,BANG CHAN,리노,이민호,LEE..."
9,ZV5IjNgpBFQ,2026-03-04T10:59:58Z,[Stray Kids : SKZ-TALKER] Ep.80,SKZ-TALKER,719482,102197,3083,[Stray Kids(스트레이 키즈) : SKZ-TALKER(슼즈토커)] Ep.80...,10,"JYP Entertainment,JYP,Stray Kids,스트레이 키즈,SKZ,스..."


In [45]:
df_videos.to_csv('../data/raw/df_videos_temporary.csv')

In [10]:
df_videos = pd.read_csv("../data/raw/df_videos_temporary.csv")
video_ids = list(df_videos['video_id'])
video_ids

['l8sSQAa1kNM',
 '9DSOFyLSv6Q',
 'Jl5nNqajrV0',
 '1tipk2X1aUo',
 'bDbzZ_Dx4Ks',
 'uB-XPe1K7SU',
 'X7jAms_f4Rc',
 'bSZL3ZZC8EE',
 'o66KJX1YJ3s',
 'ZV5IjNgpBFQ',
 'sKi1gEtnSAQ',
 'x-kl7fzEmaY',
 'h1li3NlTB0I',
 'ODoEa_1gPSc',
 'HFbirADU3_Q',
 'FEDQ14t58CA',
 'mUFKUJTMjWg',
 'PUUxE7EGP4Q',
 'OSUHqAvuSRI',
 'F6lf26nWk10',
 'Tsr3lBEn9ww',
 'IiqoozFOs1s',
 'v1xm6Pck-mo',
 'XOcH3lgw0KE',
 'bpnab0Aa5mE',
 'fETjgY83o7Q',
 'urNLPgalt6o',
 '9bHJc5Hv9Vw',
 '5Oio145WBB8',
 'nv6WtNmPXZU',
 'YsvbN665LZE',
 'ZeoHs2zfXPE',
 'wgfPkWSDUHU',
 'MI6_aH1JT7s',
 'HdgpH3_mZEs',
 'dxi4pB9b7eg',
 'uRJUVUlTevM',
 'PET396MCIV8',
 'TA7C2NBYw88',
 '3z27jOgXx-I',
 '5bzHeSphgUU',
 'ZqnBCh51qUw',
 'QNKTZSZ-X3s',
 'FR99OlxOVus',
 'xqu9eeGQxXc',
 '7oxJSrN2CNQ',
 'vPPEfLv1EpE',
 'k4IsJDwRKz4',
 'udFiiR4V6mg',
 'wIJ2TcnDKPs',
 'FCXd9nXSWTM',
 'mf967jPZrCk',
 'eHmQ_b0LkUA',
 'FECQ9MmbC4Y',
 'kaBPTrvzf8c',
 'ZSy5AcukDjI',
 'FnWKcU7ayH0',
 'Hw4Qwy_Kr-8',
 'XZL0Dvxs7jY',
 'Csaxd97bVJE',
 '1ydl2uTMPGo',
 'U9-7JBGRyMc',
 'IUreR3

# Fetching Video Transcripts

In [4]:
transcript_data = []

ytt_api = YouTubeTranscriptApi()

# Fetch the list of available transcripts for the video
available_transcript = ytt_api.list(video_ids[0])
available_transcript

In [6]:
# Find English first, fallback to Korean
transcript = available_transcript.find_transcript(['en', 'ko'])
transcript

In [7]:
# Fetch the actual transcript data chunks
transcript_chunks = transcript.fetch()
transcript_chunks

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='Yeah yeah yeah yeah', start=12.417, duration=1.625), FetchedTranscriptSnippet(text='I’m running in the rain yeah', start=14.042, duration=1.583), FetchedTranscriptSnippet(text='Hoping it washes away my broken heart ', start=15.625, duration=2.958), FetchedTranscriptSnippet(text='You once did love me, though? Right, right?', start=18.583, duration=7.334), FetchedTranscriptSnippet(text='Lover lover lover is hurt ’cause she went away', start=25.917, duration=3.416), FetchedTranscriptSnippet(text='Loved her loved her loved her so much until that rainy day', start=29.333, duration=3.334), FetchedTranscriptSnippet(text='Lover lover lover dies ’cause there’s no way', start=32.667, duration=3.25), FetchedTranscriptSnippet(text='Lover lover love is gone no matter what I say', start=35.917, duration=3.666), FetchedTranscriptSnippet(text='Oh, this is too much to handle while sober', start=39.583, duration=2.417), FetchedTranscriptSnippet(

In [9]:
type(transcript_chunks)

youtube_transcript_api._transcripts.FetchedTranscript

In [8]:
# Join the chunked dictionary elements into a single continuous text string
full_transcript = " ".join([segment.text for segment in transcript_chunks])
full_transcript

'Yeah yeah yeah yeah I’m running in the rain yeah Hoping it washes away my broken heart  You once did love me, though? Right, right? Lover lover lover is hurt ’cause she went away Loved her loved her loved her so much until that rainy day Lover lover lover dies ’cause there’s no way Lover lover love is gone no matter what I say Oh, this is too much to handle while sober I don’t want to drink no more no more no more Why’re you acting like it’s me It’s all my fault, you mean? Then again, you were always like that    Why why is it not me In your eyes I’m searching for a reason ooh ooh ooh ooh Yeah yeah yeah yeah I’m running in the rain yeah Hoping it washes away my broken heart  You once did love me, though? Right, right? Lover lover lover is hurt ’cause she went away Loved her loved her loved her so much until that rainy day Lover lover lover dies ’cause there’s no way Lover lover love is gone no matter what i say Lover lover lover needs a hug cause she‘s gone away Loved her loved her lo

In [2]:
datetime.now(timezone.utc).isoformat()

'2026-03-29T18:33:11.234492+00:00'

In [5]:
for i in range(0, 223, 5):
    print(i)

0
5
10
15
20
25
30
35
40
45
50
55
60
65
70
75
80
85
90
95
100
105
110
115
120
125
130
135
140
145
150
155
160
165
170
175
180
185
190
195
200
205
210
215
220
